In [ ]:

print(st)
for tr in st:
    if tr.stats.station not in ['B07', 'B12', 'B23']:    
        if tr.stats.channel[1]  == 'H':
            tr.data = tr.data / 3e8
        elif tr.stats.channel[1]  == 'D':
            tr.data = tr.data / 720
#st.plot(equal_scale=False)#, outfile='artemis2_launch.png');#, size=(800, 600), title='GEM Data around Launch Time');

In [ ]:
or tr in stmsfc:
    st.append(tr)
print(st)
st.write(str(SDS_DATA_PATH / 'artemis2_launch_seismic_Z.mseed'), format='MSEED')
stf = st.copy() 
stf.detrend('linear')
stf.filter('highpass', freq=0.1, corners=2, zerophase=True)
stf.select(component='Z').trim(starttime=launchtime-30, endtime=launchtime+120).plot(equal_scale=False);#, outfile='artemis2_launch_broadband.png');#, size=(800, 600), title='GEM Data around Launch Time');

In [ ]:
import matplotlib.pyplot as plt
stgood = stf.select(component='Z').trim(starttime=launchtime-30, endtime=launchtime+120)
# select B01 and B29 for comparions
stsubset = Stream(traces=[stgood.select(station='B01')[0],   stgood.select(station='B29')[0]])
stsubset.plot(equal_scale=True, outfile='artemis2_launch_broadband_near_far.png');#, size=(800, 600), title='GEM Data around Launch Time');

stsubset.plot(equal_scale=False);
stigood = stf.select(channel='DD0').trim(starttime=launchtime-30, endtime=launchtime+120)
# select B01 and B29 for comparions
stisubset = Stream(traces=[stigood.select(station='B01')[0],   stigood.select(station='B29')[0]])
stisubset.plot(equal_scale=True, outfile='artemis2_launch_infrasound_near_far.png');#, size=(800, 600), title='GEM Data around Launch Time');
from flovopy.processing.spectrograms import icewebSpectrogram
iwobj = icewebSpectrogram(stsubset)
iwobj.precompute()
iwobj.plot(dbscale=True, equal_scale=True, fmax=250.0, clim=[1e-7, 1e-4], cmap='magma', add_colorbar=False);
iwobji = icewebSpectrogram(stisubset)
iwobji.precompute()
iwobji.plot(dbscale=True, equal_scale=True, fmax=250.0, clim=[2e-2, 2e1], cmap='magma', add_colorbar=False);
from obspy import Stream
stgood = stf.select(component='Z').trim(starttime=launchtime-30, endtime=launchtime+120)
stsubset = Stream(traces=[
    stgood.select(station='B01')[0],
    stgood.select(station='B29')[0]
])
stsubset.plot(outfile = 'artemis2_launch_broadband_near_far.png')
st2 = st.copy()
st2.trim(starttime=launchtime - 60, endtime=launchtime + 240)
st2.plot(equal_scale=False, outfile=SDS_DATA_PATH.parent / 'artemis2_launch_zoom_sds.png');#, size=(800, 600), title='GEM Data around Launch Time');

import numpy as np
# split infrasound and seismic data
st2.filter("highpass", freq=0.3)
sti = Stream([tr for tr in st2 if tr.stats.channel[1:3] in ['DF', 'D0']])
sts = Stream([tr for tr in st2 if tr.stats.channel[1] == 'H'])
sts = sts.select(component='Z'  )

for this_stream, label in zip([sti, sts], ['Infrasound', 'Seismic']):
    print(f'{label} traces:')
    for tr in this_stream:
        tr.detrend(type='demean')
        if label == 'Infrasound':
            print(f'  {tr.id} - Peak amplitude: {np.abs(tr.data).max():.1f} Pa')   
        elif label == 'Seismic':
            print(f'  {tr.id} - Peak amplitude: {np.abs(tr.data).max():.1e} m/s')
    this_stream.plot(equal_scale=False, title=f'{label} Data around Launch Time', outfile = SDS_DATA_PATH.parent / f'artemis2_launch_zoom_filtered_{label.lower()}.png');#, size=(800, 600)  );



peak_amplitudes = {}
for tr in st2.copy():
    tr.detrend(type='demean')
    data = np.abs(tr.data)
    peak_amplitudes[tr.id] = data.max()
peak_amplitudes = {k: v for k, v in sorted(peak_amplitudes.items(), key=lambda item: item[1], reverse=True)}
print('Peak amplitudes for each trace (sorted):')

import pandas as pd

df = pd.DataFrame(list(peak_amplitudes.items()), columns=['Trace ID', 'Peak Amplitude'])
df.to_csv(SDS_DATA_PATH.parent / 'sds_peak_amplitudes.csv', index=False)
df['Peak Amplitude'] = df['Peak Amplitude'].round(1)
display(df)
    

In [ ]:
for tr in sti.copy():
    tr.detrend(type='demean')
    output_wav = SDS_DATA_PATH.parent / f'{tr.id}_infrasound_audio.wav'
    trace_to_audio(tr, speedup=60, output_wav=str(output_wav), audio_rate=44100, normalize=True, detrend=False, taper=True, bandpass=(0.1, 240))
    print(f'Created audio file: {output_wav}')

for tr in sts.copy():
    tr.detrend(type='demean')
    output_wav = SDS_DATA_PATH.parent / f'{tr.id}_seismic_audio.wav'
    trace_to_audio(tr, speedup=60, output_wav=str(output_wav), audio_rate=44100, normalize=True, detrend=False, taper=True, bandpass=(0.1, 240))
    print(f'Created audio file: {output_wav}')

In [ ]:
import numpy as np
from obspy import read
from scipy.io.wavfile import write as wavwrite
from scipy.signal import resample

def trace_to_audio(
    tr,
    speedup=100,
    output_wav="output.wav",
    audio_rate=44100,
    normalize=True,
    detrend=True,
    taper=True,
    bandpass=None,
):
    """
    Convert an ObsPy Trace to a WAV audio file by speeding it up.

    Parameters
    ----------
    tr : obspy.Trace
        Input seismic trace.
    speedup : float
        Factor by which to speed up playback.
    output_wav : str
        Output WAV filename.
    audio_rate : int
        Desired WAV sample rate in Hz.
    normalize : bool
        If True, scale data to int16 range.
    detrend : bool
        If True, remove mean and linear trend.
    taper : bool
        If True, apply a short taper.
    bandpass : tuple or None
        Optional (fmin, fmax) in Hz for pre-filtering the seismic data.
    """
    tr = tr.copy()

    if detrend:
        tr.detrend("demean")
        tr.detrend("linear")

    if taper:
        tr.taper(max_percentage=0.02)

    if bandpass is not None:
        fmin, fmax = bandpass
        tr.filter("bandpass", freqmin=fmin, freqmax=fmax, corners=4, zerophase=True)

    data = tr.data.astype(np.float64)

    # Remove NaNs/infs if present
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

    # Effective sampling rate after speeding up
    effective_rate = tr.stats.sampling_rate * speedup

    # Resample to desired audio rate if needed
    if abs(effective_rate - audio_rate) / audio_rate > 0.01:
        n_out = int(round(len(data) * audio_rate / effective_rate))
        if n_out <= 0:
            raise ValueError("Output sample count is invalid.")
        data = resample(data, n_out)

    if normalize:
        peak = np.max(np.abs(data))
        if peak > 0:
            data = data / peak
        data_int16 = np.int16(data * 32767)
    else:
        data_int16 = np.int16(np.clip(data, -32768, 32767))

    wavwrite(output_wav, audio_rate, data_int16)
    return output_wav